# Conformal prediction: distribution-free uncertainty

Most prediction intervals rely on the model being correct. Conformal prediction gives intervals
(for regression) and prediction sets (for classification) with a finite-sample coverage guarantee
that holds for any model and any data distribution, assuming only that the data are exchangeable.
This notebook builds split conformal prediction from scratch, verifies the coverage guarantee even
when the underlying model is badly misspecified, and shows where the guarantee stops (conditional
coverage).


## The coverage guarantee, proved

Theorem (split-conformal coverage). Let the calibration scores $s_1,\dots,s_n$ and a test score
$s_{n+1}$ be exchangeable (which holds for iid data). Set the threshold to the
$\lceil (n+1)(1-\alpha)\rceil$-th smallest calibration score. Then the prediction set
$\{y: s(x_{n+1},y)\le \hat q\}$ covers the truth with probability at least $1-\alpha$, for any model and
any distribution, at finite $n$.

Proof. By exchangeability, the rank of $s_{n+1}$ among $s_1,\dots,s_{n+1}$ is uniform on
$\{1,\dots,n+1\}$. The test point is covered exactly when its score is at most the chosen calibration
quantile, i.e. when its rank is at most $\lceil(n+1)(1-\alpha)\rceil$, which has probability
$\lceil(n+1)(1-\alpha)\rceil/(n+1)\ge 1-\alpha$. The bound uses no property of the model: a wrong model
only makes the scores, hence the intervals, larger. $\quad\blacksquare$

The applied section confirms the guarantee even for a deliberately misspecified model.

Counterexample (exchangeability is essential). Under distribution shift between calibration and test the
scores are no longer exchangeable, and coverage can fall well below $1-\alpha$. The applied exercises
break exchangeability to show the degradation; weighted conformal prediction restores coverage when the
shift is known.

## 1. Split conformal regression

Fit a model on a training split. On a held-out calibration split, record the absolute residuals.
The (1-alpha) empirical quantile of those residuals, with a small finite-sample correction, becomes
the half-width of a prediction interval that covers a new point with probability at least 1-alpha.


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
from mixle.ppl import Normal, Field, free, conformal
rng = np.random.RandomState(0)
n = 6000
x = rng.uniform(-3, 3, n); y = np.sin(x) + 0.3 * x + rng.normal(0, 0.4, n)
tr, cal, te = np.split(rng.permutation(n), [3000, 4500])
m = Normal(free * Field('x') + free, free).fit(list(y[tr]), given={'x': list(x[tr])})
alpha = 0.1
cp = conformal(m.result, y[cal], given={'x': list(x[cal])}, alpha=alpha)   # split-conformal calibration on held-out data
covered = cp.covers(y[te], given={'x': list(x[te])})
print('target coverage = %.2f' % (1 - alpha))
print('empirical coverage on test = %.3f   interval half-width = %.2f' % (covered.mean(), cp.qhat))

target coverage = 0.90
empirical coverage on test = 0.900   interval half-width = 0.92


## 2. The guarantee does not need a correct model

Split conformal coverage holds even for a deliberately wrong model, because it calibrates on the
model's own residuals. A constant predictor (always the training mean) still yields valid intervals,
just wide ones. Coverage is a property of the procedure, not of model quality.


In [2]:
one = np.ones(n)
cm = Normal(free * Field('one'), free).fit(list(y[tr]), given={'one': list(one[tr])})   # constant-mean (misspecified)
cp0 = conformal(cm.result, y[cal], given={'one': list(one[cal])}, alpha=alpha)
cov0 = cp0.covers(y[te], given={'one': list(one[te])})
print('constant (misspecified) model: empirical coverage = %.3f  half-width = %.2f' % (cov0.mean(), cp0.qhat))
print('coverage is still ~0.90; the price of a bad model is a wide interval, not a broken guarantee.')

constant (misspecified) model: empirical coverage = 0.903  half-width = 1.80
coverage is still ~0.90; the price of a bad model is a wide interval, not a broken guarantee.


## 3. Conformal prediction sets for classification

For classification the nonconformity score is one minus the predicted probability of the true class.
The calibrated threshold defines, for each test point, the set of labels whose score is small enough.
The set has guaranteed marginal coverage and grows when the model is unsure.


In [3]:
from mixle.ppl import ConformalClassifier
from mixle.stats import MultivariateGaussianEstimator
from mixle.inference.estimation import optimize
K, d = 4, 6
rc = np.random.RandomState(1)
centers = rc.normal(0, 1.3, (K, d)); nC = 4000
yc = rc.randint(0, K, nC); Xc = centers[yc] + rc.normal(0, 1.5, (nC, d))
itr, ical, ite = np.split(rc.permutation(nC), [2000, 3000])
# a mixle generative classifier: one Gaussian per class + class priors -> posterior p(y|x)
comps = [optimize([list(r) for r in Xc[itr][yc[itr] == k]], MultivariateGaussianEstimator(dim=d),
                  max_its=1, rng=rc, print_iter=1000) for k in range(K)]
logpri = np.log(np.bincount(yc[itr], minlength=K) / len(itr))
def posteriors(X):
    Xl = [list(r) for r in X]
    ll = np.stack([c.seq_log_density(c.dist_to_encoder().seq_encode(Xl)) for c in comps], 1)
    z = ll + logpri; z -= z.max(1, keepdims=True); P = np.exp(z); return P / P.sum(1, keepdims=True)
cc = ConformalClassifier(posteriors(Xc[ical]), yc[ical], alpha=alpha)   # nonconformity = 1 - p(true class)
Pte = posteriors(Xc[ite])
print('target coverage = %.2f   empirical set coverage = %.3f' % (1 - alpha, cc.covers(Pte, yc[ite]).mean()))
print('average prediction-set size = %.2f labels (1 = confident, >1 = hedging)' % cc.set_sizes(Pte).mean())

target coverage = 0.90   empirical set coverage = 0.900
average prediction-set size = 1.37 labels (1 = confident, >1 = hedging)


## 4. Marginal coverage is not conditional coverage

The guarantee is marginal: averaged over all points it holds, but coverage can be too low in hard
regions and too high in easy ones. We bin the regression test points by absolute x and show coverage
varies, which motivates locally adaptive variants.


In [4]:
abs_x = np.abs(x[te]); bins = np.quantile(abs_x, [0, .33, .66, 1.0])
for lo, hi in zip(bins[:-1], bins[1:]):
    mask = (abs_x >= lo) & (abs_x <= hi)
    print('|x| in [%.1f, %.1f]: coverage = %.3f' % (lo, hi, covered[mask].mean()))
print('constant-width intervals over-cover the easy middle and under-cover the wiggly edges;')
print('conformalized quantile regression (next) and Mondrian conformal target conditional coverage.')

|x| in [0.0, 1.0]: coverage = 0.937
|x| in [1.0, 2.0]: coverage = 0.869
|x| in [2.0, 3.0]: coverage = 0.894
constant-width intervals over-cover the easy middle and under-cover the wiggly edges;
conformalized quantile regression (next) and Mondrian conformal target conditional coverage.


## 5. Conformalized quantile regression

The constant-width band above spends its width poorly: too wide where the data is calm, too narrow where it is volatile, so its coverage is uneven across x. Conformalized quantile regression (Romano, Patterson, Candes 2019) fixes this by calibrating a quantile-regression band rather than a point predictor. Fit a lower and an upper conditional quantile, then conformalize the score $E_i = \max(q_{lo}(x_i) - y_i,\ y_i - q_{hi}(x_i))$; the band $[q_{lo}(x) - \hat q,\ q_{hi}(x) + \hat q]$ keeps exact marginal coverage while inheriting the adaptive, heteroscedastic width of the quantiles, so coverage holds within each stratum of x where the constant-width band fails.

In [5]:
from mixle.ppl import ConformalQuantileRegressor
rq = np.random.RandomState(1)
nh = 6000
xh = rq.uniform(0, 5, nh); yh = 2 + 1.5 * xh + rq.normal(0, 0.3 + 0.7 * xh, nh)   # noise grows with x
htr, hcal, hte = np.split(rq.permutation(nh), [3000, 4500])
qlo = Normal(free * Field('x') + free, free).fit(list(yh[htr]), given={'x': list(xh[htr])}, quantile=0.05)
qhi = Normal(free * Field('x') + free, free).fit(list(yh[htr]), given={'x': list(xh[htr])}, quantile=0.95)
cqr = ConformalQuantileRegressor(qlo.result, qhi.result, yh[hcal], given={'x': list(xh[hcal])}, alpha=0.1)
cov_cqr = cqr.covers(yh[hte], given={'x': list(xh[hte])}); lo_h, hi_h = cqr.interval({'x': list(xh[hte])})
mean_h = Normal(free * Field('x') + free, free).fit(list(yh[htr]), given={'x': list(xh[htr])})
cov_const = conformal(mean_h.result, yh[hcal], given={'x': list(xh[hcal])}, alpha=0.1).covers(yh[hte], given={'x': list(xh[hte])})
print('CQR marginal coverage = %.3f   (target 0.90)' % cov_cqr.mean())
xt = np.asarray(xh[hte]); bins = np.quantile(xt, [0, .33, .66, 1.0])
print('conditional coverage by tercile of x:')
for a, b in zip(bins[:-1], bins[1:]):
    m = (xt >= a) & (xt <= b)
    print('  x in [%.1f, %.1f]:  CQR = %.3f   constant-width = %.3f' % (a, b, cov_cqr[m].mean(), cov_const[m].mean()))
print('CQR keeps coverage in every stratum with an adaptive width (%.1f at x<1 -> %.1f at x>4);' %
      ((hi_h - lo_h)[xt < 1].mean(), (hi_h - lo_h)[xt > 4].mean()))
print('the constant-width band over-covers where the noise is small and under-covers where it is large.')

CQR marginal coverage = 0.917   (target 0.90)
conditional coverage by tercile of x:
  x in [0.0, 1.6]:  CQR = 0.923   constant-width = 1.000
  x in [1.6, 3.4]:  CQR = 0.915   constant-width = 0.947
  x in [3.4, 5.0]:  CQR = 0.914   constant-width = 0.776
CQR keeps coverage in every stratum with an adaptive width (2.3 at x<1 -> 11.4 at x>4);
the constant-width band over-covers where the noise is small and under-covers where it is large.


## References

- Vovk, V., Gammerman, A. & Shafer, G. (2005). Algorithmic Learning in a Random World. Springer.
- Lei, J., G'Sell, M., Rinaldo, A., Tibshirani, R. & Wasserman, L. (2018). Distribution-free predictive inference for regression. JASA. https://doi.org/10.1080/01621459.2017.1307116
- Romano, Y., Patterson, E. & Candes, E. (2019). Conformalized quantile regression. NeurIPS.
- Angelopoulos, A. & Bates, S. (2023). Conformal prediction: a gentle introduction. Foundations and Trends in Machine Learning. https://arxiv.org/abs/2107.07511


## Exercises

1. Implement conformalized quantile regression: fit a quantile regressor for the lower and upper quantiles, then conformalize the residuals to restore exact marginal coverage. Compare interval widths to the constant-width baseline across the range of x.
2. Vary alpha from 0.2 down to 0.01 and plot the empirical coverage and mean interval width. Confirm coverage tracks the target and width grows as alpha shrinks.
3. Build a Mondrian (group-conditional) conformal predictor that calibrates separately within strata of x, and check whether conditional coverage improves.
4. Break exchangeability by introducing distribution shift between calibration and test (shift the mean of x at test time) and measure how far coverage degrades.
